# Notes

Currently time series work even with duplicate measurement, but they are all on different graphs. Leaving that for now. Need to do soil profiles and check soil moisture and snow depth across sites.

In [4]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import os
import matplotlib.cm as cm
from io import StringIO
import sys
import nc_time_axis
import os

sys.path.append('C:/Users/madel/Code/GeoNorth')

from utils import *

In [5]:
sites = ("TVC", "SC", "BC", "HC", "IQ")

def is_not_empty_listdir(path):
    return not len(os.listdir(path)) == 0

In [6]:
# Full Loop

for site in sites:
    
    # pull ctsm file
    ctsm_file = f"..\\CTSM_data\\run_1\\merged_files\\{site}_1995-2014.nc"
    df = xr.open_dataset(ctsm_file, engine="netcdf4")

    # starting with TSOI
    vari = "TSOI"

    if is_not_empty_listdir(f"C:\\Users\\madel\\Code\\GeoNorth\\Observational_data\\{site}_obs\\raw\\{vari}"):
        ds = xr.open_dataset(f"../Observational_data/{site}_obs/processed/{site}_{vari}.nc")


        # sort for variables
        ds_ctsm = df[f"{vari}"]
        ds_obs = ds[f"{vari}"]
        ds_obs = ds_obs.sortby("depth")

        # Interpolate ctsm data
        da_interp = interpolate_to_depth(df, ds_obs.depth)

        # Plots
        final_plot_tsoi(ds_obs, da_interp, f"{site}", ds_obs.depth, f"{vari}")
        august_profile_tsoi(ds_obs, ds_ctsm, site, ds_obs.depth, f"{vari}")

        # want to use the best data if there are multiple observations at depth
        #later could be updated to combine all data at depth into one but comes with issues like "is this run okay data or not"
        used = clean_august_profile_tsoi(ds_obs, ds_ctsm, site, ds_obs.depth, f"{vari}")
        single_seasonal_cycle_tsoi(ds_obs, da_interp, site, used, vari)
        seasonal_plot_tsoi(ds_obs, da_interp, site, ds_obs.depth, f"{vari}")
        
    
    # next H2OSOI
    vari = "H2OSOI"

    if is_not_empty_listdir(f"C:\\Users\\madel\\Code\\GeoNorth\\Observational_data\\{site}_obs\\raw\\{vari}"):
        ds = xr.open_dataset(f"../Observational_data/{site}_obs/processed/{site}_{vari}.nc")

        # sort for variables
        ds_ctsm = df[f"{vari}"]
        ds_obs = ds[f"{vari}"]
        ds_obs = ds_obs.sortby("depth")

        # plot
        final_plot_h2osoi(ds_obs, df, f"{site}", ds_obs.depth, f"{vari}")
        seasonal_plot_h2osoi_full(ds_obs, df, f"{site}", ds_obs.depth, f"{vari}")

    # next snow depth
    vari = "SNOW_DEPTH"
    if is_not_empty_listdir(f"C:\\Users\\madel\\Code\\GeoNorth\\Observational_data\\{site}_obs\\raw\\{vari}"):
        ds = xr.open_dataset(f"../Observational_data/{site}_obs/processed/{site}_{vari}.nc")
        era = xr.open_dataset(f"C:/Users/madel/Code/GeoNorth/ERA5/{vari}/processed/{site}/{site}_ERA5_SNOW_DEPTH.nc")

        # sort for variables
        ds_ctsm = df[f"{vari}"]
        ds_obs = ds[f"{vari}"]
        ds_obs = ds_obs.sortby("depth")
        era5_sd = era["sde"]

        final_plot_sd(ds_obs, ds_ctsm, era5_sd, f"{site}", ds_obs.depth, f"{vari}")
    



C:\Users\madel\AppData\Local\Temp\ipykernel_25252\3127479028.py:7: FutureWarning: In a future version, xarray will not decode the variable 'SNOW_PERSISTENCE' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  df = xr.open_dataset(ctsm_file, engine="netcdf4")
C:\Users/madel/Code/GeoNorth\utils.py:163: FutureWarning: In a future version of xarray to_datetimeindex will default to returning a 'us'-resolution DatetimeIndex instead of a 'ns'-resolution DatetimeIndex. This warning can be silenced by explicitly passing the `time_unit` keyword argument.
  obs = obs.assign_c

# Metrics for Later

In [7]:
def compute_metrics(obs_concat, sim):
    obs = obs_concat.set_index("time").to_xarray()
    obs_daily = obs.resample(time="1D").mean()
    sim_C = sim - 273.15
    
    # Strip timezone info if present
    obs_daily["time"] = obs_daily["time"].values.astype("datetime64[ns]")
    sim_C["time"] = sim_C["time"].values.astype("datetime64[ns]")
    
    obs_aligned, sim_aligned = xr.align(obs_daily, sim_C, join="inner")
    
    obs_vals = obs_aligned["obs_t"].values
    sim_vals = sim_aligned.values
    valid = ~np.isnan(obs_vals) & ~np.isnan(sim_vals)
    rmse = np.sqrt(np.nanmean((sim_vals[valid] - obs_vals[valid])**2))
    bias = np.nanmean(sim_vals[valid] - obs_vals[valid])
    return rmse, bias